# Poll Results Visualizer - Exploratory Data Analysis
Step-by-step walkthrough of the pipeline: **load -> clean -> analyse -> visualise -> insights**.

Run `python main.py` once first (it creates the raw CSV), or the first cell will generate data for you.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # make `src` importable from notebooks/

import pandas as pd
from src import config, analysis, visualization
from src.data_generator import generate_poll_data
from src.data_cleaning import clean_poll_data

pd.set_option('display.width', 140)

## 1. Load raw data (screenshot this: *dataset preview*)

In [ ]:
raw = pd.read_csv(config.RAW_DATA_FILE) if config.RAW_DATA_FILE.exists() else generate_poll_data()
print(raw.shape)
raw.head(10)

## 2. Data-quality check before cleaning

In [ ]:
print('Missing values per column:')
print(raw.isna().sum())
print()
print('Duplicate respondent+question rows:', raw.duplicated(['respondent_id', 'question_id']).sum())
print('Region spellings found:', sorted(raw['region'].dropna().unique()))

## 3. Clean (screenshot this: *cleaned dataset preview*)

In [ ]:
df, log = clean_poll_data(raw)
df.head(10)

In [ ]:
print('Region values after cleaning:', sorted(df['region'].unique()))
df.info()

## 4. EDA - who answered?

In [ ]:
people = df.drop_duplicates('respondent_id')
for col in ['region', 'age_group', 'gender']:
    print(people[col].value_counts(normalize=True).mul(100).round(1).rename('% respondents'))
    print()

In [ ]:
people.groupby(people['response_date'].dt.to_period('W'))['respondent_id'].count().rename('respondents per week')

## 5. Vote / share summary (screenshot this: *summary table*)

In [ ]:
analysis.option_share(df, 'Q1').sort_values('rank')

In [ ]:
analysis.lead_analysis(df, 'Q1')

## 6. Region and demographic comparison

In [ ]:
analysis.segment_share(df, 'Q1', 'region')

In [ ]:
analysis.segment_leaders(df, 'Q1', 'age_group')

In [ ]:
analysis.satisfaction_score(df, 'Q2', by='region')

In [ ]:
pd.DataFrame([analysis.association_test(df, 'Q1', by) for by in ['region', 'age_group', 'gender']])

## 7. Trend over time

In [ ]:
analysis.trend_share(df, 'Q1')

## 8. Charts

In [ ]:
from IPython.display import Image, display
for p in visualization.create_all_charts(df):
    display(Image(filename=str(p), width=750))

## 9. Insights

In [ ]:
from src.insights import generate_insights, recommendations
for i, s in enumerate(generate_insights(df), 1):
    print(f'{i}. {s}')
print()
print('Recommendations:')
for r in recommendations(df):
    print('-', r)